# How do you know your agent works?
### Evaluating Earth Observation agents: benchmarks, metrics, and a reusable harness

Hands-on session, BIFOLD Agentic AI for Earth Observation workshop. Pi School and ESA Φ-lab, EVE project.

Building an agent is straightforward. Showing that it works reliably is harder.

### What changes compared with evaluating an LLM

Scoring a plain LLM is mostly one question: *is the answer right?* The model gets a prompt and returns text, so you compare that text with a reference (accuracy, F1, BLEU, a judge). An **agent** is different:

- **It acts, not only answers.** It plans, calls tools (raster readers, index calculators, GIS operations, map APIs, code), reads the results, and decides what to do next. The answer is the last step of a longer process.
- **A right answer can hide a bad process.** The agent can guess, take a lucky shortcut, or waste ten calls. A wrong answer can come from one bad step in an otherwise sound run. The final answer alone cannot tell these apart.
- **The path is part of the result.** Which tools it chose, in which order, with which arguments, how many calls, and how it recovered from errors are all measurable, but only if the run is recorded as an **execution trace**.
- **Runs are not repeatable by default.** The same agent can take a different path on the same question, so you need fixed tasks, fixed tools, recorded configurations, and often several runs.
- **The environment is part of the system.** Tool schemas, data, the model endpoint, and timeouts change the score, so a number is only meaningful with the setup that produced it.

That is why agent evaluation has two kinds of metrics: **end-to-end** metrics (did it get the result?) and **step-by-step** metrics (did it get there the right way?).

### How this notebook is organised

- **Part I, metrics.** Step-by-step metrics (trajectories, Tool-Any-Order, Tool-In-Order, Tool-Exact-Match, Parameter Accuracy, efficiency) and end-to-end metrics (accuracy, Pass@k, LLM-as-a-judge).
- **Part II, EO benchmarks for agents.** What the existing benchmarks ask, what they score, and which ones this harness runs.
- **Part III, the evaluation framework.** How the `agentic_evals` harness runs a benchmark, then a real Earth-Bench run: assemble an agent, score it, diagnose failures from traces, add a judge, and put your own agent on a leaderboard next to the baseline.

The model under test is EVE, served through an OpenAI-compatible endpoint. Any other endpoint works the same way.

## Setup

This folder is a closed environment. `setup.sh` created `.venv` and `.env` here. Select the kernel **agentic-evals workshop** and fill in `EVE_API_KEY`, `EVE_BASE_URL`, and `MODEL_NAME` in `.env` before continuing.

The next cell points data and run output at this folder, then loads `.env`. If you restart the kernel, run it again.

In [1]:
from pathlib import Path
import os, sys

root = Path.cwd()
if not (root / "workshop.py").is_file():
    candidate = root / "sagemaker"
    if (candidate / "workshop.py").is_file():
        root = candidate
    else:
        raise RuntimeError("Run this notebook with the working directory set to the sagemaker folder.")
os.chdir(root)
sys.path.insert(0, str(root))
sys.modules.pop("workshop", None)   # re-running this cell picks up edits to workshop.py
if Path(sys.prefix).resolve() != (root / ".venv").resolve():
    raise RuntimeError(f"Wrong kernel ({sys.executable}). Select the kernel 'agentic-evals workshop'.")

from dotenv import load_dotenv
load_dotenv(root / ".env")
if not os.environ.get("EVE_BASE_URL", "").strip() and os.environ.get("EVE_API_BASE_URL", "").strip():
    os.environ["EVE_BASE_URL"] = os.environ["EVE_API_BASE_URL"].strip()

os.environ["AGENTIC_EVALS_DATA"] = str((root / "data").resolve())
os.environ["AGENTIC_EVALS_RUNS_DIR"] = str((root / "runs").resolve())
# Tool servers write their banners, warnings and tracebacks to runs/.../kit_stderr.log, not here.
os.environ["AGENTIC_EVALS_KIT_STDERR"] = "file"
# With W&B off, turn the inspect-wandb hooks off too. Otherwise they log a warning on every event.
if os.environ.get("WANDB_MODE", "").strip() == "disabled":
    os.environ["INSPECT_WANDB_MODELS_ENABLED"] = "false"
    os.environ["INSPECT_WANDB_WEAVE_ENABLED"] = "false"

assert sys.version_info >= (3, 12), f"Python 3.12+ needed, found {sys.version.split()[0]}"
if "agentic_evals.paths" in sys.modules:
    from agentic_evals.paths import DATA_DIR
    expected = (root / "data").resolve()
    if DATA_DIR != expected:
        raise RuntimeError("Restart the kernel so AGENTIC_EVALS_DATA is picked up.")

REPO = root.parent   # the agentic-evals checkout this folder belongs to
print("folder", root)
print("python", sys.executable)

folder /Users/antoniolopez/pischool/agentic-evals/sagemaker
python /Users/antoniolopez/pischool/agentic-evals/sagemaker/.venv/bin/python


In [2]:
import pandas as pd
from workshop import (
    WORKSHOP_QUESTIONS, check_environment, download_earth_bench, judge, leaderboard, results_df, run,
)
from agentic_evals.agents.langgraph_react import langgraph_agent

pd.set_option("display.precision", 2)
MODEL_NAME = os.environ["MODEL_NAME"].strip()
MODEL = f"openai-api/eve/{MODEL_NAME}"
print("model under test", MODEL)
print("workshop questions", list(WORKSHOP_QUESTIONS))
check_environment()

model under test openai-api/eve/jsc/alias-eve
workshop questions [11, 2, 7, 8, 19, 21, 26, 31, 36, 42]
python 3.12.13
executable /Users/antoniolopez/pischool/agentic-evals/sagemaker/.venv/bin/python
data /Users/antoniolopez/pischool/agentic-evals/sagemaker/data/earth_bench
runs /Users/antoniolopez/pischool/agentic-evals/sagemaker/runs
tool kits Analysis, Index, Inversion, Perception, Statistics
perception lookup True
rasters on disk for 10/10 workshop questions
endpoint identified True 


token budget available


## Part I. Metrics for agents

Agent metrics come in two kinds, named after what they read:

| | Step-by-step metrics | End-to-end metrics |
| --- | --- | --- |
| **Reads** | the trajectory: every tool call and its arguments | the finished run: the final answer and how the run ended |
| **Question** | Did it get there the right way? | Did it get the right result, and how reliably? |
| **Needs** | an expert reference trajectory | a gold answer, or a judge |
| **Metrics** | Tool-Any-Order, Tool-In-Order, Tool-Exact-Match, Parameter Accuracy, step accuracy, efficiency, tool errors | accuracy, Pass@k, LLM-as-a-judge, termination |

Earth-Bench and ThinkGeo both name their metrics this way.

### I.1 Step-by-step metrics

Step-by-step metrics score the **process**: which tools the agent called, in which order, with which arguments, and how many calls it needed.

#### What is a trajectory?

A **trajectory** is the ordered list of tool calls an agent made: for each call, the tool's name and its arguments, and (in some benchmarks) the observation the tool returned. A benchmark that scores trajectories ships a **reference trajectory** written and checked by experts, and compares the agent's list with it. A real example, **Earth-Bench question 11**, shown as the expert's three calls in order:

![The gold trajectory of Earth-Bench question 11: get_filelist, lst_multi_channel, calculate_threshold_ratio](figures/trajectory_q11.png)

Other benchmarks record the same thing with small differences: a GeoBenchX task can have several reference solutions, and for an unsolvable task the reference is the single step `reject_task()` (refusing is the correct behaviour). EO-Gym also stores the **observation** returned by each step.

The point of recording this: two agents can give the same final answer through very different trajectories, and an agent can follow the expert path and still answer wrong. Only the trajectory shows which.

#### The metrics

Let $t^*$ be the expert tool list with $m$ steps and $t$ the agent's list. Earth-Bench ([Feng et al., ICLR 2026](https://arxiv.org/abs/2509.23141), Appendix B.2) defines them like this, and the other benchmarks use close variants:

- **Tool-Any-Order (TAO).** The share of expert tools that appear anywhere in the agent's list, $\dfrac{\lvert \mathrm{Set}(t^*) \cap \mathrm{Set}(t) \rvert}{\lvert \mathrm{Set}(t^*) \rvert}$. Did the agent find the right tools?
- **Tool-In-Order (TIO).** Walk through the expert tools in order and look for each one later in the agent's list than the previous match. Count matches until one is missing, divide by $m$. Extra calls in between are allowed. Did it use them in the right order?
- **Tool-Exact-Match (TEM).** The longest common prefix of $t$ and $t^*$, divided by $m$. No extra calls allowed. Did it follow the expert path with no detours?
- **Parameter Accuracy (ParamAcc).** Like TEM, but each call must also have the expert's arguments. Did it call them correctly? The next part explains how.
- **Step accuracy** (ThinkGeo). Instruction, tool, argument and summary correctness judged for each step of the run.

Five runs of the same three-step task, scored with the harness's own functions. Pick a run in the dropdown to see its trace next to the expert's:

In [3]:
import pandas as pd
from agentic_evals.tasks.earth_bench import metrics

def call(name, **kw):
    return {"name": name, "input": kw}

gold = [
    call("get_filelist", dir_path="benchmark/data/question11"),
    call("lst_multi_channel", band31_path="b31.tif", band32_path="b32.tif", output_path="lst.tif"),
    call("calculate_threshold_ratio", image_paths=["lst.tif"], threshold=310.0),
]
runs = {
    "A) The expert path":              gold,
    "B) An extra call first":          [gold[0], gold[0], gold[1], gold[2]],
    "C) Right tools, wrong threshold": [gold[0], gold[1], call("calculate_threshold_ratio", image_paths=["lst.tif"], threshold=300.0)],
    "D) Skips get_filelist":           [gold[1], gold[2]],
    "E) A different tool in step 2":   [gold[0], call("band_ratio", band_a="b31.tif", band_b="b32.tif"), gold[2]],
}
from IPython.display import display
from workshop import compare_widget

table = pd.DataFrame(
    {
        name: {
            "TAO": metrics.tool_any_order(pred, gold),
            "TIO": metrics.tool_in_order(pred, gold),
            "TEM": metrics.tool_exact_match(pred, gold),
            "ParamAcc": metrics.parameter_accuracy(pred, gold),
            "calls": len(pred),
        }
        for name, pred in runs.items()
    }
).T.round(2)
display(table)
compare_widget(runs, gold)

,TAO,TIO,TEM,ParamAcc,calls
A) The expert path,1.00,1.00,1.00,1.00,3.0
B) An extra call first,1.00,1.00,0.33,0.33,4.0
"C) Right tools, wrong threshold",1.00,1.00,1.00,0.67,3.0
D) Skips get_filelist,0.67,0.00,0.00,0.00,2.0
E) A different tool in step 2,0.67,0.33,0.33,0.33,3.0


In the trace view, green rows match the expert, amber rows use the right tool with different arguments, red rows are a different or missing tool, and grey rows are extra calls. Wavy underlines mark the arguments that differ. Read the table row by row:

- **B** wastes one call. TAO and TIO forgive it; TEM and ParamAcc drop to 1/3 because every later step is shifted by one.
- **C** uses the right tools in the right order, so TAO, TIO and TEM are all 1. Only ParamAcc sees the wrong threshold (2/3).
- **D** never lists the files. TIO is 0 because it looks for the first expert tool first and stops when it is missing.
- **E** picked another tool in step 2, which is the case the next part is about.

#### How parameter accuracy is computed

If the agent called a different tool, what do you compare the arguments with? **You do not.** The tool name is checked first. Arguments only matter once the tool matches. How the two are lined up differs between benchmarks:

- **Earth-Bench (and the step metrics we add for GeoBenchX): position by position.** Call *i* of the agent is compared with step *i* of the expert. The step counts only if the tool name is the same **and** the arguments are exactly equal (every key and value: file paths, thresholds, band order). At the first step that fails either test, scoring stops, and later steps do not count even if they would match. ParamAcc is the number of correct steps divided by $m$, so it can never be higher than TEM. A wrong tool at step *i* simply makes step *i* wrong: there is nothing to compare its arguments with.
- **EO-Gym: align by tool name, then compare.** For each expert step in order, scan forward in the agent's calls for the first one with the **same tool name**. If there is none, that step scores 0 and no arguments are compared. If there is one, its arguments are compared after normalisation (keys lowercased and sorted, whitespace collapsed, paths lowercased with slashes and cache ids normalised, numbers equal within 1e-6) and its returned observation is compared too. Both must match for the step to count. Extra calls and a call to some other tool are skipped over, not punished.

Run E under both rules. For the comparison the agent's later calls also differ in spelling (upper and lower case in a path, a cache id, a float changed at the 7th decimal), to show EO-Gym's normalisation:

In [4]:
from IPython.display import HTML, display
from agentic_evals.tasks.eo_gym import metrics as eo_gym
from workshop import rules_html

# Run E, written as EO-Gym steps (name, arguments, observation). The agent calls band_ratio in step 2,
# and spells the later paths, cache id and float slightly differently from the expert.
expected = [
    eo_gym.Step("get_filelist", {"dir_path": "d"}, '["a.tif"]'),
    eo_gym.Step("lst_multi_channel", {"band31_path": "A.tif", "output_path": "o/lst_ab12cd34ef56.tif"}, "saved"),
    eo_gym.Step("calculate_threshold_ratio", {"threshold": 310.0}, "49.7"),
]
actual = [
    eo_gym.Step("get_filelist", {"dir_path": "d"}, '["a.tif"]'),
    eo_gym.Step("band_ratio", {"x": 1}, "0.3"),
    eo_gym.Step("lst_multi_channel", {"band31_path": "a.tif", "output_path": "o/LST_00000000aaaa.tif"}, "saved"),
    eo_gym.Step("calculate_threshold_ratio", {"threshold": 310.0000001}, "49.7"),
]
display(HTML(rules_html(expected, actual)))

Same agent behaviour, different outcome: Earth-Bench's ParamAcc is 0.33 because scoring stops at the wrong tool in step 2, and EO-Gym's parameters score is 1.0 because it skips the detour and normalises spelling. Both are strict in a way you should know about: an equally valid call with a different output file name is a mismatch. That is one reason benchmarks also use judges.

#### Efficiency, errors and termination

- **Efficiency / length.** Earth-Bench reports $\lvert t \rvert / \lvert t^* \rvert$, a ratio (above 1 means more calls than the expert). GeoBenchX reports a signed relative difference, $(n - \bar{m}) / \bar{m}$.
- **Error rate.** The share of tool calls that fail (EO-Gym's function-call error rate; our `tool_errors` count) or of generated code that does not run (UnivEARTH).
- **Termination.** Did the run finish, or hit a step or model-call limit? This harness records it on every sample.

One real run, with each of these numbers:

![Cost and reliability of one Earth-Bench run: efficiency, tool errors, termination](figures/run_anatomy.png)

### I.2 End-to-end metrics

End-to-end metrics score the **finished run**, without looking at how it got there: is the answer right, how often across attempts, and did the run end normally (termination, above).

#### Accuracy / exact match

The extracted answer equals the gold one. Earth-Bench reads the letter from `<Answer>C<Answer>` and compares it with the gold option. Cheap and objective, but a right answer says nothing about the path, and four options mean a blind guess is right 25% of the time.

#### Pass@k

An agent is stochastic, so one run is one sample of its behaviour. **Pass@k is the probability that at least one of k attempts at a task is correct**, averaged over tasks. Pass@1 is the reliability of a single try; a large gap between Pass@1 and Pass@k means the agent *can* solve the task but not *consistently*.

Think of all possible runs on one task as a **tree of traces**: at each step the agent picks a tool, reads the result and branches again, and each path ends in a right or a wrong answer. The numbers on the edges are the **probability of taking that branch**. Multiply them along a path to get the probability of that trace, and add the green leaves to get $p$, the chance that one attempt is correct. With $k$ independent attempts, Pass@k $= 1 - (1-p)^k$.

![Tree of possible traces for question 11, with three sampled attempts and the resulting Pass@1, Pass@2 and Pass@3](figures/pass_at_k_tree.png)

You cannot read the edge numbers off a real agent, so you estimate: run $n$ attempts per task, count the $c$ correct ones, and use the unbiased estimator (Chen et al., 2021)

$$\text{Pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

Pass@k costs k times the tokens of one run. In this harness each attempt is one Inspect *epoch*.

#### LLM-as-a-judge

Exact match fails on free-text answers and on tasks with several valid solution paths. A **judge** is a language model that is given the question, the gold answer and a candidate answer, and returns a verdict. Benchmarks use it for the official answer score: GeoBenchX (the judge rates a whole solution as no match, partial or match), EO-Gym (same meaning, yes or no) and ThinkGeo. The judge model is part of the number, so always report it. Section III.9 runs two judges on one answer.

A judge is only as good as its reply format. This is how the harness reads GeoBenchX's:

In [5]:
from agentic_evals.tasks.geobenchx import metrics as geobenchx

reply = (
    "<MATCH REASONING>The candidate loads the same dataset and filters the same countries, "
    "but colours the map by a different column.</MATCH REASONING>\n<MATCH SCORE>1</MATCH SCORE>"
)
print(geobenchx.parse_verdict(reply))                  # 0 no match, 1 partial, 2 match
print(geobenchx.parse_verdict("Looks right to me!"))   # no tag: the sample is left unscored, not guessed

(1, 'The candidate loads the same dataset and filters the same countries, but colours the map by a different column.')
(None, 'judge output has no <MATCH SCORE> tag')


What to remember about judges (section III.9 runs two on one answer):

- **The verdict depends on the judge model.** Scores from different judges are not comparable, so the harness records the judge with every verdict and keeps each judge as its own result.
- **Judges can disagree with the metrics and with each other.** `agentic_evals.scorers.agreement` reports raw agreement and Cohen's kappa between any two verdicts stored in a log.
- **They cost a model call per sample** and can favour answers that look like their own style.

## Part II. EO benchmarks for agents

A benchmark is a set of tasks with gold answers (and sometimes gold trajectories) plus a published scoring protocol. They differ in what the agent does, which tools it has, and which of the metrics above they report.

### II.1 Overview

| Benchmark | Focus | Metrics | In harness |
| --- | --- | --- | --- |
| **Earth-Bench** (Feng et al., ICLR 2026) | Raster analysis with 104 fixed tools | Accuracy; TAO, TIO, TEM, ParamAcc, efficiency | yes |
| **GeoBenchX** (Krechetova & Kochedykov, SIGSPATIAL 2025) | GIS workflows, including tasks that must be rejected | LLM-judge match / partial / no match; length difference | yes |
| **EO-Gym** (arXiv:2605.01250) | Multimodal tasks with imagery, tool server | Pass@k with LLM judge; binary tool metrics; parameters | in progress |
| **OpenEarthAgent** | Tool-based EO agents | to be added | in progress |
| **ThinkGeo** (Shabbir et al., 2025) | Optical and SAR remote sensing, 14 tools | Judge-based answer accuracy; instruction, tool, argument, summary accuracy | no |
| **UnivEARTH** (Kao et al., 2025) | Yes/no questions solved with Earth Engine code | Accuracy; code failure rate | no |
| **GeoCode-Bench** (Hou et al., 2024) | Geospatial code generation | Answer accuracy | no |
| **MapEval-API** (Dihan et al., ICML 2025) | Map-API questions: places, routes, distances | Accuracy | no |

Three things to take from this table:

1. **Every benchmark brings its own metrics.** This harness implements each benchmark's published metrics and does not mix them into one shared score.
2. **The same name can mean different things.** "Tool exact match" is a fraction in Earth-Bench and a 0/1 flag in EO-Gym (section II.3).
3. **LLM judges are part of some protocols.** For GeoBenchX, EO-Gym and ThinkGeo the official answer score comes from a judge model, so the judge must be reported with the number.

To pick a benchmark, start from the capability you care about: raster analysis with fixed tools (Earth-Bench), GIS workflows and knowing when to refuse (GeoBenchX), visual grounding with imagery (EO-Gym, ThinkGeo), code against a real data platform (UnivEARTH, GeoCode-Bench), or map services (MapEval-API).

### II.2 Three benchmarks in this harness

**Earth-Bench** ([Feng et al., ICLR 2026](https://arxiv.org/abs/2509.23141)), the one we run in Part III:

- **What it is:** 248 multiple-choice Earth Observation questions that an agent answers by calling tools on real satellite data.
- **Data:** three modalities: Spectrum (questions 1 to 100), Products (101 to 188) and RGB (189 to 248). We use Spectrum.
- **Tools:** 104 tools in five MCP kits (Analysis, Index, Inversion, Perception, Statistics): file listing, land surface temperature, vegetation indices, thresholds, statistics.
- **Two regimes:** in **AP** (auto-planning) the agent chooses its own steps; in **IF** (instruction-following) the question lists them. Always report them separately. We use AP.
- **Gold standard:** a correct option, plus an expert list of tool calls.
- **Metrics:** accuracy (end-to-end) and TAO, TIO, TEM, ParamAcc and efficiency (step-by-step), Appendix B.2. The harness computes them with `agentic_evals.tasks.earth_bench.metrics`, which follows upstream `evaluate/step_by_step.py` and is tested against the published score files.

**GeoBenchX** (Krechetova & Kochedykov, SIGSPATIAL 2025):

- **What it is:** 202 GIS tasks on statistical, vector and raster data, with 24 tools. Task groups: merge, process, spatial and heatmaps.
- **Task kinds:** 120 solvable, 79 unsolvable (the right behaviour is to reject), 3 control questions that need no tool.
- **Gold standard:** one or more reference solutions per task, as tool calls.
- **Metrics:** an LLM judge compares the whole solution with the references (no match, partial, match, reported as shares), plus the length difference. The step metrics we compute are labelled as our addition.

**EO-Gym** (arXiv:2605.01250), port in progress; its metric functions are already in the harness:

- **What it is:** a multimodal, interactive environment: imagery is in the prompt, tools sit behind a server, and the test set has 1,436 questions.
- **Attempts:** three independent trajectories per question.
- **Metrics:** Pass@k with a judge on the final answer, 0/1 tool metrics per trajectory, and a parameters score that compares arguments and tool outputs.
- **Needs** a vision-capable model.

### II.3 Same names, different definitions

Each benchmark keeps its own metric module, so the same name can mean different things. Run B from section I.1 (one extra call first), scored with **EO-Gym**'s and **GeoBenchX**'s published definitions:

In [6]:
from agentic_evals.tasks.eo_gym import metrics as eo_gym
from agentic_evals.tasks.geobenchx import metrics as geobenchx

pred = runs["B) An extra call first"]
gold_names, pred_names = metrics.tool_names(gold), metrics.tool_names(pred)
pd.DataFrame(
    {
        "Earth-Bench (fraction)": [metrics.tool_any_order(pred, gold), metrics.tool_in_order(pred, gold),
                                   metrics.tool_exact_match(pred, gold)],
        "EO-Gym (0/1 per trajectory)": [int(eo_gym.tools_any_order(gold_names, pred_names)),
                                        int(eo_gym.tools_in_order(gold_names, pred_names)),
                                        int(gold_names == pred_names)],
    },
    index=["any order", "in order", "exact match"],
)

,Earth-Bench (fraction),EO-Gym (0/1 per trajectory)
any order,1.00,1
in order,1.00,1
exact match,0.33,0


In [7]:
# Length: Earth-Bench reports a ratio, GeoBenchX a signed relative difference.
print("Earth-Bench efficiency  |t| / |t*|          =", round(metrics.efficiency(pred, gold), 3))
print("GeoBenchX length_diff   (n - mean_ref)/mean_ref =", round(geobenchx.length_diff(len(pred), [gold]), 3))

# EO-Gym answers are judged per trajectory and summarised as Pass@k.
# 3 trajectories, 1 correct: unbiased Pass@1 and Pass@3 (Chen et al., 2021).
print("EO-Gym unbiased Pass@1 =", round(eo_gym.unbiased_pass_at_k(n=3, c=1, k=1), 3),
      "  Pass@3 =", eo_gym.unbiased_pass_at_k(n=3, c=1, k=3))

Earth-Bench efficiency  |t| / |t*|          = 1.333
GeoBenchX length_diff   (n - mean_ref)/mean_ref = 0.333
EO-Gym unbiased Pass@1 = 0.333   Pass@3 = 1.0


## Part III. The evaluation framework, and a real Earth-Bench run

Parts I and II said what to measure and with which benchmarks. This part is how: the `agentic_evals` harness, then a real Earth-Bench run from end to end.

### III.1 What the harness does and what is implemented

The harness runs an agent on a public EO benchmark under the same conditions as every other agent, records **both** the final answer and the tool-call trajectory, and scores the run with **that benchmark's own published metrics**. It is built on [Inspect AI](https://inspect.aisi.org.uk/); agents are LangGraph graphs; tools are reached over [MCP](https://modelcontextprotocol.io).

| Benchmark | Task name | Scored with | Status |
| --- | --- | --- | --- |
| Earth-Bench | `agentic_evals/earth_bench` | Accuracy; TAO, TIO, TEM, ParamAcc, efficiency | ready |
| GeoBenchX | `agentic_evals/geobenchx` | Judge match shares (no match, partial, match) and length difference; step metrics as our labelled addition | ready |
| EO-Gym | `agentic_evals/eo_gym` | Pass@k, judge-based correctness, binary tool metrics, parameters score | port in progress |
| OpenEarthAgent | not yet | not yet | in progress (tools only so far) |

Around the benchmarks, the harness provides:

- **Agents.** `langgraph_react` is the baseline ReAct agent. `langgraph_agent(graph)` evaluates **any LangGraph graph** you write, which is what you do in this notebook. Both run through Inspect's `agent_bridge()`, so the agent is routed to whichever model the eval names.
- **Tools.** Each benchmark's tools run as MCP servers. A tool error is returned to the model instead of crashing the run, and every tool call is recorded as a **trace** (name, arguments, output, error).
- **Scorers.** One module per benchmark with the metrics from its paper (`tasks/<benchmark>/metrics.py`, `scorers.py`). The metric functions are tested against the published results or the reference scorer. A judge that is not part of a paper's protocol is an extra scorer with its own result; verdicts are never combined into a shared score.
- **Logs.** Every run is an Inspect `.eval` log with all messages, tool calls, scores, and the identity of the model endpoint. Logs can be browsed (`inspect view`) and **rescored offline** without calling the agent again.
- **Configuration.** A run is a YAML file in `configs/`: model, endpoint, task arguments, generation settings, which samples, epochs. Runs are repeatable and nothing is hard-coded.
- **Optional.** Weights & Biases logging of runs, off by default in this notebook.

### III.2 One evaluation is one Inspect Task

```
dataset   questions, gold answers, gold trajectories          (tasks/<benchmark>/dataset.py)
setup     record the endpoint, start this sample's MCP tool servers
solver    your agent: a LangGraph graph run through agent_bridge() (agents/)
scorers   the benchmark's published metrics, plus optional extra judges (tasks/<benchmark>/scorers.py)
   |
   +--> .eval log: every message, tool call, score, and the model/endpoint identity
```

Let us build the Earth-Bench task object and look inside:

In [8]:
from inspect_ai._util.registry import registry_info
from agentic_evals.tasks.earth_bench.task import earth_bench

task = earth_bench(regime="ap", modality="spectrum")
print("samples :", len(task.dataset), "(AP, Spectrum)")
print("setup   :", [s.__qualname__.split(".")[0] for s in task.setup])
print("solver  :", registry_info(task.solver).name, "(default; we pass our own below)")
print("scorers :", [registry_info(s).name for s in task.scorer])
print("metrics the paper reports:", task.metadata["published_metrics"])

samples : 100 (AP, Spectrum)
setup   : ['record_run_identity', 'skip_unavailable_rgb', 'publish_mcp_servers']
solver  : agentic_evals/langgraph_react (default; we pass our own below)
scorers : ['agentic_evals/earth_bench_scorer']
metrics the paper reports: ['accuracy', 'efficiency', 'tao', 'tio', 'tem', 'param_acc']


The agent never sees benchmark code. It gets the sample's messages and whatever MCP tools the task published. That is why the same agent can run on Earth-Bench and GeoBenchX (and on EO-Gym once its port is done), and why the scorer, not the agent, decides what counts as correct.

### III.3 Running it

A run is described in a YAML file in `configs/`, so it can be repeated exactly. This is the CLI version of the baseline you will run in section III.7:

In [9]:
print((REPO / "configs" / "workshop_earth_bench_eve.yaml").read_text())

# Earth-Bench workshop baseline: the ten Spectrum auto-planning questions of
# sagemaker/workshop.py (WORKSHOP_QUESTIONS), EVE through its OpenAI-compatible
# endpoint, the default LangGraph ReAct solver.
#
# This is the same run the hands-on notebook makes with workshop.run(), written
# as a run config. Earth-Bench does not publish this subset; see workshop.py.
#
# Inspect reads EVE_API_KEY and EVE_BASE_URL from .env in the working directory.
# Rasters for these questions:
#   uv run python scripts/setup_benchmark.py earth-bench --questions 11,2,7,8,19,21,26,31,36,42
# Then:
#   uv run inspect eval --run-config configs/workshop_earth_bench_eve.yaml

task:
  task: agentic_evals/earth_bench
  args:
    regime: ap
    modality: spectrum

model:
  model: openai-api/eve/jsc/alias-eve
  args:
    strict_tools: false

generate_config:
  # Upstream Earth-Agent runners use temperature 0.1 (also the task default).
  temperature: 0.1
  # An unset max_tokens makes this endpoint reserve the rest o

From a terminal at the repository root:

```bash
uv run inspect eval --run-config configs/workshop_earth_bench_eve.yaml   # run
uv run inspect view                                                     # browse transcripts, traces, scores
uv run inspect score logs/<run>.eval --scorer agentic_evals/earth_bench_scorer   # rescore offline
```

In this notebook we call the same thing from Python (`inspect_ai.eval`), because the agent graph is defined here. `workshop.run()` is a thin wrapper that fixes the task, the question set, and the timeouts.

### III.4 The Earth-Bench workshop set

#### Questions

`WORKSHOP_QUESTIONS` is our workshop subset of Earth-Bench:

- **10 Spectrum tasks** in the AP regime, all with a gold trajectory.
- **3 tool calls each**, except question 8, which has 4.
- **Different retrieval methods** (TVDI, LST, band ratio, TES, TTM, split-window, MODIS, ATI), so the agent cannot reuse one recipe.
- **Not an official split.** Earth-Bench does not define this subset; we picked it so the session stays short.

Each question is multiple choice and comes with its gold tool trajectory.

In [10]:
from agentic_evals.tasks.earth_bench.dataset import load_samples

ap = {s.id: s for s in load_samples("ap", "spectrum") if s.id in WORKSHOP_QUESTIONS}
sample = ap[11]

system_prompt, question = sample.input.split("Question:", 1)
print("SYSTEM PROMPT (sent as part of the user message)", system_prompt.strip(), sep="\n")
print("\nQUESTION (AP), with the data folder appended and the choices", question.strip(), sep="\n")
print("\nGold answer:", sample.target)
print("\nGold trajectory:")
for i, c in enumerate(sample.metadata["gold_tool_calls"], 1):
    print(f"  {i}. {c['name']}({c['input']})")

SYSTEM PROMPT (sent as part of the user message)
You are a geoscientist, and you need to use tools to answer multiple-choice questions about Earth observation data analysis. Note that if a tool returns an error, you can only try again once. Ultimately, you only need to explicitly tell me the correct choice.
ATTENTION:
1. When a tool returns "Result saved at /path/to/file", you must use the full returned path "/path/to/file" in all subsequent tool calls.
2. For each question, you must provide the choice you think is most appropriate.Don't gibe me another format. Your final answer format must be:
<Answer>Your choice<Answer>

QUESTION (AP), with the data folder appended and the choices
Based on thermal Band 31 and Band 32 data over Tokyo metropolitan area  on August 10, 2022, calculate the proportion of the area with land surface temperature exceeding 310 K.benchmark/data/question11
A.14.36%
B.27.63%
C.49.74%
D.62.47%

Gold answer: C

Gold trajectory:
  1. get_filelist({'dir_path': 'bench

In [11]:
# The same question in the IF regime: the steps are written in the question.
IF = {s.id: s for s in load_samples("if", "spectrum") if s.id in WORKSHOP_QUESTIONS}
print(IF[11].input.split("Question:", 1)[1].strip())

Based on thermal Band 31 and Band 32 data over the Tokyo metropolitan area on August 10, 2022, first list the input TIFF files, then estimate land surface temperature (LST) using the split-window method, and finally calculate the proportion of the area where LST exceeds 310 K.benchmark/data/question11
A.14.36%
B.27.63%
C.49.74%
D.62.47%


#### Data

The rasters come from the public Hugging Face dataset `Sssunset/Earth-Bench` (the full benchmark is 16.6 GB). We download only the folders for our 10 questions, into this folder's `data/`.

In [12]:
raster_root = download_earth_bench(WORKSHOP_QUESTIONS)
files = [p for p in raster_root.rglob("*") if p.is_file() and ".cache" not in p.parts]
print(f"{len(files)} files in {raster_root}  ({sum(p.stat().st_size for p in files) / 1e6:.1f} MB)")
print(sorted(p.name for p in raster_root.iterdir() if not p.name.startswith(".")))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 590 files:   0%|          | 0/590 [00:00<?, ?it/s]

Rasters for 10 tasks in /Users/antoniolopez/pischool/agentic-evals/sagemaker/data/earth_bench/benchmark/data


590 files in /Users/antoniolopez/pischool/agentic-evals/sagemaker/data/earth_bench/benchmark/data  (1490.5 MB)
['question11', 'question19', 'question2', 'question21', 'question26', 'question31', 'question36', 'question42', 'question7', 'question8']


#### Tools

The agent does not call Python functions directly. Earth-Bench's tools run as five [MCP](https://modelcontextprotocol.io) servers (stdio), one per kit. Any MCP client can use them, which is the same idea as EVE's tool registry.

In [13]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from agentic_evals.tasks.earth_bench.tools import KITS, kit_servers

# Each kit starts its own server process. Its diagnostics go to runs/demo/kit_stderr.log.
client = MultiServerMCPClient(kit_servers(Path("runs/demo").resolve()))
for kit in KITS:
    kit_tools = await client.get_tools(server_name=kit)
    print(f"{kit:<11} {len(kit_tools):>3} tools   e.g. {', '.join(t.name for t in kit_tools[:4])}")

Analysis     10 tools   e.g. compute_linear_trend, mann_kendall_test, sens_slope, stl_decompose


Index        12 tools   e.g. calculate_batch_ndvi, calculate_batch_ndwi, calculate_batch_ndbi, calculate_batch_evi


Inversion    17 tools   e.g. band_ratio, lst_single_channel, lst_multi_channel, split_window


Perception   15 tools   e.g. threshold_segmentation, bbox_expansion, count_above_threshold, count_skeleton_contours


Statistics   50 tools   e.g. coefficient_of_variation, skewness, kurtosis, calc_batch_image_mean


In [14]:
# Call one tool by hand, the way the agent will.
tools = await client.get_tools()
by_name = {t.name: t for t in tools}
print(len(tools), "tools in total")
print(await by_name["get_filelist"].ainvoke({"dir_path": "benchmark/data/question11"}))

104 tools in total


[{'type': 'text', 'text': '["2022_08_10_0110_BT_31.tif","2022_08_10_0110_BT_32.tif"]', 'id': 'lc_474d5dd0-a568-4c62-a899-741a28b51fba'}]


Every model call carries all these tool descriptions, which is a lot of tokens. Remember this when you design your own graph: a longer loop costs more.

### III.5 The agent: ReAct

ReAct ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) alternates *reasoning*, *acting* (a tool call) and *observing* (the tool result) until the model answers.

The graph is defined in the next cell. `create_agent` is LangChain 1.x's ReAct graph. Two pieces of middleware sit on it: one repairs Mistral tool calls that arrive as text, and one moves images out of tool messages, because chat-completions APIs accept images only on user messages.

`langgraph_agent(react_graph)` is the harness around the graph. It starts the MCP servers, routes `llm` to your model, gives tool errors back to the model instead of crashing, and records every tool call as a trace. The scorer reads that trace. You do not edit the harness.

In [ ]:
from langchain.agents import create_agent
from agentic_evals.agents.langgraph_react import NormalizeMistralToolCalls, ToolImagesAsUserMessage

def react_graph(llm, tools):
    return create_agent(
        llm,
        tools=tools,
        middleware=[NormalizeMistralToolCalls(), ToolImagesAsUserMessage()],
        name="workshop_react",
    )

### III.6 One question, with the raw Inspect call

`workshop.run()`, used later, is a thin wrapper around this call. `strict_tools=False` is required: some Earth-Bench tool schemas have untyped arguments that strict mode rejects.

In [ ]:
from inspect_ai import eval as inspect_eval
from workshop import check_budget

check_budget()   # one-token request: stops here if the model's token budget is spent

one = inspect_eval(
    earth_bench(regime="ap", modality="spectrum", solver=langgraph_agent(react_graph, name="workshop_react")),
    model=MODEL,
    model_args={"strict_tools": False},
    sample_id=[11],
    log_dir="logs/one",
    display="none",
    max_tokens=4096,          # cap each reply
    attempt_timeout=180,      # give up on a silent call after 3 minutes
    stream_idle_timeout=60,
    max_retries=6,            # back off and retry on HTTP 429 (the room shares one endpoint)
    max_connections=2,        # at most 2 model calls in flight from this notebook
)[0]
print("status:", one.status)

In [ ]:
import json
from agentic_evals.agents.trace_callback import read_trace

def show_trace(log, question):
    s = next(s for s in log.samples if int(s.id) == int(question))
    for e in read_trace(s.store):
        print(f"{e['step']:>2}. {e['name']}({json.dumps(e['input'])[:140]})")
        print("    ->", str(e.get("output"))[:120], "  [ERROR]" if e.get("error") else "")
    print("\nFinal message:", s.output.completion[-200:])

show_trace(one, 11)

This is the **execution trace**: every tool call with its arguments and output, in order. Everything that follows (trajectory metrics, the judge, your diagnosis) is computed from it.

#### The score of that sample

In [ ]:
score = one.samples[0].scores["earth_bench_scorer"]
md_ = score.metadata

print({k: round(v, 3) for k, v in score.value.items()})
print("\nextracted answer :", score.answer, "| gold:", one.samples[0].target)
print("gold tools       :", md_["gold_tools"])
print("predicted tools  :", md_["predicted_tools"])
print("first divergence :", md_["first_divergence_step"])
for step in md_["step_matches"]:
    print("  ", step)

`tool_errors` counts calls that raised an error or returned one (for example a made-up file path). `perception_failures` should stay at 0 here; if it does not, the setup is broken, not the model.

### III.7 All 10 questions

This runs up to 4 samples in parallel and takes a few minutes. Each question costs about 120k tokens of your model budget, because every model call carries the descriptions of all 104 tools.

In [ ]:
baseline = run(
    langgraph_agent(react_graph, name="workshop_react"),
    model=MODEL,
    questions=WORKSHOP_QUESTIONS,
    log_dir="logs/baseline",
    max_samples=4,
)
df_base = results_df(baseline)
df_base[["accuracy", "tao", "tio", "tem", "param_acc", "efficiency", "tool_errors", "n_calls", "termination"]]

In [ ]:
COLS = ["accuracy", "tao", "tio", "tem", "param_acc", "efficiency", "tool_errors"]
print("Baseline, mean over", len(df_base), "questions (Earth-Bench AP, Spectrum)")
df_base[COLS].mean().round(3)

With 10 questions, one answer moves accuracy by 10 points. Treat differences of a few points as noise; the point of this session is the method, not the ranking.

Look at the rows where accuracy and the trajectory metrics disagree:

- **right answer, low TEM**: the agent found another route, or got lucky;
- **wrong answer, high TIO**: right tools, wrong arguments or a wrong reading of the result.

An answer-only score cannot tell these cases apart. That is the reason trajectory metrics exist.

In [ ]:
df_base.assign(
    right_answer=df_base.accuracy == 1,
    expert_path=df_base.tem == 1,
)[["accuracy", "tio", "tem", "param_acc", "right_answer", "expert_path"]]

### III.8 Diagnose a failure from its trace

Pick the question where the agent's path was furthest from the expert's, and read its trace next to the gold trajectory.

In [ ]:
from IPython.display import HTML, display
from workshop import compare_html, sample_calls

worst = df_base.sort_values(["tio", "accuracy"]).index[0]
print("Question", worst, df_base.loc[worst, ["accuracy", "tao", "tio", "tem", "param_acc"]].round(2).to_dict())
print("Gold:")
for i, c in enumerate(ap[worst].metadata["gold_tool_calls"], 1):
    print(f"  {i}. {c['name']}({json.dumps(c['input'])[:140]})")
print("\nAgent:")
show_trace(baseline, worst)

# The same side-by-side view as in section I.1, for the real run.
agent_calls = sample_calls(next(x for x in baseline.samples if int(x.id) == int(worst)))
gold_calls = ap[worst].metadata["gold_tool_calls"]
display(HTML(compare_html(agent_calls, gold_calls, f"Question {worst}")))

Questions to ask while you read:

- Did the agent choose a different but valid tool, or a wrong one?
- Did it get the arguments wrong (paths, thresholds, band order)?
- Did it hit a tool error and recover, or give up? The tool server's own traceback is in `runs/earth_bench/question<N>/<epoch-id>/kit_stderr.log`.
- Is the final answer right even though the trajectory differs from the expert's? What does that say about each metric?

The `.eval` log keeps the full transcript. `inspect view` (run from this folder: `.venv/bin/inspect view --log-dir logs`) shows every model message next to every tool call.

### III.9 Two judges, one answer

Earth-Bench answers are option letters, so an exact match is enough there. A free-text answer needs a judge. Below, the candidate answer is free text, and the gold is option C of question 11 (49.74%). Two judges from different model families judge it against the gold with the same prompt: EVE-Instruct on the workshop key, and GPT-4.1 mini through OpenRouter. Each returns score 1 (same meaning) or 0 (different meaning). When they disagree, the score depends on the judge, which is why both are shown.

In [16]:
import re

from IPython.display import HTML, display

from workshop import JUDGES, judge_answer, judges_html

question = ap[11].input.split("Question:", 1)[1]
stem = re.split(r"\nA\.", question)[0].replace("benchmark/data/question11", "").strip()
gold = "C. " + re.search(r"^C\.(.+)$", question, re.M).group(1).strip()
candidate = "About 50 percent of the area is above 310 K, so the answer is C."

verdicts = [await judge_answer(model, stem, gold, candidate, base_url=url) for model, url in JUDGES]
display(HTML(judges_html(stem, gold, candidate, verdicts)))

inspect_ai v0.3.277
- hooks enabled: 3
  agentic_evals/langfuse_scores: Attach each sample's scores to its Langfuse trace.
  agentic_evals/uncapped_max_tokens: Treat --max-tokens -1 as no output cap.
  agentic_evals/wandb_compare: Log shared summary metrics and trace tables onto the W&B run.

┌────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────┐
│ /Users/antoniolopez/pischool/agentic-evals/sagemaker/.venv/lib/python3.12/site-packages/inspect_ai/model/_mode │
│ l.py:1650 in generate                                                                                          │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
AttemptTimeoutError: attempt_timeout '120' exceeded.

Read the two cards against the gold. The judges see only the question, the gold and the candidate, not the agent's trace. Limits to keep in mind:

- **The verdict depends on the judge model.** Report the judge with every score, and do not compare scores from different judges.
- **Judges disagree.** When they do, neither is the answer key; check the reasons and the gold yourself.
- **Each call costs tokens** on the model budget, and a judge can favour answers in its own style.

### III.10 Your agent on the leaderboard

Any LangGraph graph can be evaluated, as long as it follows this contract. `langgraph_agent(factory)` handles the rest (tools, model routing, trace, scoring).

- Your **factory** is a function `(llm, tools) -> compiled graph`.
- `llm` is already routed to your model. Call `llm.bind_tools(tools)` yourself if your nodes need tools.
- The graph state must have a `messages` key (use `MessagesState`). The first message already contains the system prompt and the question.
- The **last LLM call** must produce the final answer in the format `<Answer>B<Answer>`. A node that edits the answer after the last LLM call will not be scored.
- Tool calls are traced automatically, in any node or subgraph.

Here is a ReAct graph written by hand, node by node. It behaves like the baseline, without the two middleware pieces from section III.5.

In [ ]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

def my_react_graph(llm, tools):
    llm_with_tools = llm.bind_tools(tools)

    async def agent_node(state: MessagesState):
        return {"messages": [await llm_with_tools.ainvoke(state["messages"])]}

    g = StateGraph(MessagesState)
    g.add_node("agent", agent_node)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)   # tool call -> "tools", else end
    g.add_edge("tools", "agent")
    return g.compile()

#### Exercise

Start from `my_react_graph`, change one thing, and see what the metrics say. Pick one idea, or bring your own:

1. **Plan first.** Add a node before the loop that asks the LLM for a numbered plan using the tool names, then run ReAct with the plan in context.
2. **Check the format.** After the loop, add a node that asks the LLM to re-read its answer and output only `<Answer>X<Answer>`. (It must be an LLM call, see the contract above.)
3. **Limit retries.** Add a counter in the state and stop calling tools after N errors.
4. **Change the prompt.** Add a short system message with advice for EO tools (check file lists first, reuse returned paths).

Give your agent a name: it is the row label on the leaderboard. Test on 2 or 3 questions first, then run all 10 so the run counts.

In [ ]:
AGENT_NAME = "my_agent"   # change me: your team or idea, e.g. "plan_first_team3"

def my_graph(llm, tools):
    # TODO: start from my_react_graph and change it
    return my_react_graph(llm, tools)

mine = run(
    langgraph_agent(my_graph, recursion_limit=50, name=AGENT_NAME),
    model=MODEL,
    questions=WORKSHOP_QUESTIONS[:3],      # use WORKSHOP_QUESTIONS for the leaderboard run
    log_dir="logs/mine",
    max_samples=3,
)
df_mine = results_df(mine)
df_mine[COLS + ["n_calls"]]

In [ ]:
# Compare on the same questions only.
same = df_base.index.intersection(df_mine.index)
pd.concat(
    {"baseline": df_base.loc[same, COLS].mean(), AGENT_NAME: df_mine.loc[same, COLS].mean()}, axis=1
).round(3)

Did your change fix the failure you found in section III.8, or move it somewhere else? Look at trajectories, not only accuracy.

#### The leaderboard

`leaderboard()` reads every finished run in `logs/` that covered **exactly** the ten workshop questions, one row per run. It does not invent a combined score: the columns are Earth-Bench's own metrics, sorted by accuracy, and judge verdicts (if a run has them) are separate columns. AP and IF are separate rows.

For a shared room leaderboard, everyone writes runs to one log directory (Inspect accepts `s3://` paths) and calls `leaderboard(<that directory>)`.

In [ ]:
leaderboard("logs")

## Wrap-up

What you did:

1. Surveyed EO-agent benchmarks and saw that each one scores something different, with its own definitions.
2. Separated answer-only from trajectory evaluation, and computed Accuracy, TAO, TIO, TEM and ParamAcc.
3. Ran an agent through a harness that records every tool call, and read the trace of a failure.
4. Added an LLM judge by rescoring a saved log, and saw where it disagrees with the metrics.
5. Wrote your own graph and compared it with the baseline on identical tasks.

### Single agent vs multi-agent

*Coming soon.* The same harness will run a multi-agent system (an orchestrator with specialist agents) on the same ten questions, so it lands on the same leaderboard as a single agent. The trace already records which agent made each call.

### Next steps

- Run the other regime (`regime="if"`) and report AP and IF separately.
- Try more questions or other Earth-Bench modalities (`download_earth_bench` with a products question list needs about 12 GB).
- Run GeoBenchX through the same harness (see the repository README) and watch the metrics change meaning. EO-Gym and OpenEarthAgent are in progress.
- Write an MCP server for your own EO service and contribute it to the EVE tool registry.

### Troubleshooting

- `401` or `403` from the model: check `EVE_API_KEY` and `EVE_BASE_URL` in this folder's `.env`.
- `404 model not found`, or `check_environment` says the endpoint does not list the model: check `MODEL_NAME`. The check is `GET {EVE_BASE_URL}/models`, the same call the harness records on each sample.
- `BudgetExceeded`: your key's token budget is spent; the message says when it resets. Retrying will not help; ask the organizers for a key with budget. A full pass of this notebook uses about 1.8M tokens.
- `RateLimitError` or HTTP 429: the endpoint is busy. `run()` already backs off and retries; if it persists, lower `max_samples` and wait a minute.
- `Wrong kernel` in the setup cell, or `ModuleNotFoundError`: select the kernel **agentic-evals workshop** (top right in Jupyter and VS Code).
- Many `tool_errors`: read the trace with `show_trace`. Usually the model invented a file path.
- A run is very slow: lower `max_samples` or test on `WORKSHOP_QUESTIONS[:3]`.
- Your run is missing from the leaderboard: it must finish and cover all ten questions.
- Import errors after install: restart the kernel and run the setup cell again.

### References

- Feng et al. *Earth-Agent: Unlocking the Full Landscape of Earth Observation with Agents* (Earth-Bench). ICLR 2026. [arXiv:2509.23141](https://arxiv.org/abs/2509.23141)
- Krechetova & Kochedykov. *GeoBenchX: Benchmarking LLMs in Agent Solving Multistep Geospatial Tasks.* SIGSPATIAL 2025. [arXiv:2503.18129](https://arxiv.org/abs/2503.18129)
- *EO-Gym: A Multimodal, Interactive Environment for Earth Observation Agents.* [arXiv:2605.01250](https://arxiv.org/abs/2605.01250)
- Shabbir et al. *ThinkGeo: Evaluating Tool-Augmented Agents for Remote Sensing Tasks.* [arXiv:2505.23752](https://arxiv.org/abs/2505.23752)
- Kao et al. *Towards LLM Agents for Earth Observation* (UnivEARTH). [arXiv:2504.12110](https://arxiv.org/abs/2504.12110)
- Hou et al. *Can Large Language Models Generate Geospatial Code?* (GeoCode-Bench). [arXiv:2410.09738](https://arxiv.org/abs/2410.09738)
- Dihan et al. *MapEval: A Map-Based Evaluation of Geo-Spatial Reasoning in Foundation Models.* ICML 2025. [arXiv:2501.00316](https://arxiv.org/abs/2501.00316)
- Yao et al. *ReAct: Synergizing Reasoning and Acting in Language Models.* [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)
- Chen et al. *Evaluating Large Language Models Trained on Code* (Pass@k). [arXiv:2107.03374](https://arxiv.org/abs/2107.03374)
- Atrio et al. *EVE: A Domain-Specific LLM Framework for Earth Intelligence.* ACL 2026.
- [Inspect AI](https://inspect.aisi.org.uk/), [LangGraph](https://github.com/langchain-ai/langgraph), [Model Context Protocol](https://modelcontextprotocol.io), [agentevals](https://github.com/langchain-ai/agentevals)